# ***1) Correct Stable (but generate new image after each new click on generate button)***

In [ ]:
!pip install -U diffusers==0.31.0 transformers==4.46.0 accelerate safetensors

In [ ]:
import gradio as gr
import torch
from diffusers import StableDiffusionInpaintPipeline
import cv2
import numpy as np
from PIL import Image

# ----------------- Load Model -----------------
device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

pipe.enable_attention_slicing()

# ----------------- Crack Mask Generator -----------------
def generate_mask(img):
    np_img = np.array(img.convert("RGB"))[:, :, ::-1]
    gray = cv2.cvtColor(np_img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 20, 80)

    mask = cv2.dilate(edges, np.ones((5,5), np.uint8), iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((7,7), np.uint8))
    mask = cv2.medianBlur(mask, 7)

    return Image.fromarray(mask).convert("L")

# ----------------- Restoration Function -----------------
def restore_artwork(uploaded_img, prompt_strength):
    if uploaded_img is None:
        return None, None

    # speed optimization
    uploaded_img = uploaded_img.resize((512, 512))

    mask = generate_mask(uploaded_img)

    prompt = "restore original painting, remove cracks, maintain artistic style"
    negative_prompt = "cartoon, distorted, unrealistic"

    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=uploaded_img,
        mask_image=mask,
        guidance_scale=7.5,
        strength=prompt_strength,
        num_inference_steps=20
    ).images[0]

    return result, mask

# ----------------- GRADIO INTERFACE (API ENABLED) -----------------
demo = gr.Interface(
    fn=restore_artwork,
    inputs=[
        gr.Image(type="pil", label="Upload Image"),
        gr.Slider(0.1, 0.6, value=0.25, step=0.05, label="Strength")
    ],
    outputs=[
        gr.Image(label="Restored Image"),
        gr.Image(label="Mask")
    ],
    title="🎨 Artwork Restoration (Stable Diffusion)"
)

demo.launch(share=True)